# Experiment 001B v2.2: Multi-Device / Multi-Precision Gemma Probe on Colab

This notebook is based on Experiment 001B v2.1. It keeps the profiling loop and plotting algorithm intact, while refactoring the configuration, model loading, and metric adaptation layers to support GPU/CPU and FP16/INT8/INT4 experiments.

Manual decode is still used instead of `model.generate()` so `requested_gen_len` is controlled as an exact decode-step count.

## 1. 实验配置

使用交互式控件选择设备、精度和测试矩阵。请先运行本 cell，并点击“✅ 确认配置”，再继续运行后续 cell。

In [ ]:
#@title 1.1 实验配置 / Experiment Controls
# 中文注释：本 cell 只负责实验配置，不加载模型、不运行推理。
# 设计意图：把设备、精度、prompt/gen 矩阵和重复次数统一放到入口处，减少后续手工改代码造成的变量污染。
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as exc:
    raise ImportError('缺少 ipywidgets。请先运行依赖安装 cell，或安装 ipywidgets 后重新运行本 cell。') from exc

# 中文注释：保留 v2.1 的模型选择能力，方便 Exp001B 继续做不同 Gemma 2 规模的压力测试。
model_choice_widget = widgets.Dropdown(
    options=['Gemma 2 2B IT', 'Gemma 2 9B IT', 'Gemma 2 27B IT'],
    value='Gemma 2 2B IT',
    description='模型 Model',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='420px'),
)

# 中文注释：设备模式决定模型和输入张量放在 CUDA GPU 还是 CPU。
device_mode_widget = widgets.Dropdown(
    options=['GPU (CUDA)', 'CPU Only'],
    value='GPU (CUDA)',
    description='设备模式 Device Mode',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='420px'),
)

# 中文注释：推理精度用于控制 FP16 / INT8 / INT4；CPU 下量化不会真正启用，会在加载阶段回退到 FP32。
precision_widget = widgets.Dropdown(
    options=['FP16', 'INT8', 'INT4'],
    value='FP16',
    description='推理精度 Precision',
    style={'description_width': '160px'},
    layout=widgets.Layout(width='420px'),
)

# 中文注释：SelectMultiple 支持按住 Ctrl / Command 多选，避免用字符串手写矩阵。
prompt_lengths_widget = widgets.SelectMultiple(
    options=[64, 128, 256, 512, 1024, 2048],
    value=(64, 128, 256, 512),
    description='Prompt Lengths（按住Ctrl多选）',
    style={'description_width': '220px'},
    layout=widgets.Layout(width='540px', height='140px'),
)

gen_lengths_widget = widgets.SelectMultiple(
    options=[32, 64, 128, 256],
    value=(32, 64, 128),
    description='Generation Lengths（按住Ctrl多选）',
    style={'description_width': '240px'},
    layout=widgets.Layout(width='560px', height='120px'),
)

# 中文注释：重复次数固定为少数选项，避免误设过大导致 Colab 运行时间失控。
repeat_widget = widgets.Dropdown(
    options=[1, 3, 5],
    value=3,
    description='每组重复次数（取均值）',
    style={'description_width': '180px'},
    layout=widgets.Layout(width='380px'),
)

confirm_button = widgets.Button(
    description='✅ 确认配置',
    button_style='success',
    tooltip='将当前控件值写入后续 cell 使用的全局变量',
    layout=widgets.Layout(width='180px'),
)
config_output = widgets.Output()

# 中文注释：这些变量是后续 cell 的配置入口；先给默认值，避免用户未点击按钮时变量不存在。
CONFIG_CONFIRMED = False
DEVICE_MODE = 'gpu'
PRECISION = 'fp16'
prompt_lengths = [64, 128, 256, 512]
gen_lengths = [32, 64, 128]
REPEAT = 3
model_choice = 'Gemma 2 2B IT'
save_results = True


def sync_config_from_widgets(show=True):
    # 中文注释：把 widget 状态同步成普通 Python 变量，供后续 cell 稳定读取。
    global CONFIG_CONFIRMED, DEVICE_MODE, PRECISION, prompt_lengths, gen_lengths, REPEAT, model_choice

    model_choice = model_choice_widget.value
    DEVICE_MODE = 'gpu' if device_mode_widget.value == 'GPU (CUDA)' else 'cpu'
    PRECISION = precision_widget.value.lower()
    prompt_lengths = [int(x) for x in prompt_lengths_widget.value]
    gen_lengths = [int(x) for x in gen_lengths_widget.value]
    REPEAT = int(repeat_widget.value)
    CONFIG_CONFIRMED = True

    if show:
        with config_output:
            config_output.clear_output()
            print('✅ 当前配置已确认：')
            print(f'   设备模式：{device_mode_widget.value}')
            print(f'   推理精度：{precision_widget.value}')
            print(f'   Prompt Lengths：{prompt_lengths}')
            print(f'   Gen Lengths：{gen_lengths}')
            print(f'   每组重复次数：{REPEAT}')
            print(f'   模型：{model_choice}')
            if DEVICE_MODE == 'cpu':
                print('⚠️ CPU模式下建议先用小矩阵（prompt=[64], gen=[32]）验证跑通')
            if DEVICE_MODE == 'cpu' and PRECISION in ('int8', 'int4'):
                print('⚠️ CPU量化支持有限，加载阶段会以FP32加载')


confirm_button.on_click(lambda _: sync_config_from_widgets(show=True))

# 中文注释：初始化默认配置；如果用户修改控件，需要再次点击确认按钮。
sync_config_from_widgets(show=False)

display(widgets.VBox([
    model_choice_widget,
    device_mode_widget,
    precision_widget,
    widgets.HBox([prompt_lengths_widget, gen_lengths_widget]),
    repeat_widget,
    confirm_button,
    config_output,
]))
print('请检查配置并点击“✅ 确认配置”。修改控件后请重新点击确认，再运行后续 cell。')

## 2. Drive Paths

Mount Google Drive and locate the project, model cache, temporary CSV directory, and temporary figure directory. Experiment 001B temporary outputs are written under `results/exp001B/temp/`.

In [ ]:
#@title 2.1 Mount Google Drive and Set Paths
# 中文注释：本 cell 负责挂载 Google Drive、定位项目目录，并显式修复/设置当前工作目录。
# 设计意图：Colab 有时会因为 Drive 重挂载或目录被移动而出现“当前工作目录缺失”，先切回 /content 再切到项目根目录可以避免后续相对导入异常。
from pathlib import Path
import os
import sys

try:
    os.chdir('/content')
except Exception as exc:
    print('Reset working directory to /content skipped:', exc)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Google Drive mount skipped or failed:', exc)

PROJECT_DIR = Path('/content/drive/MyDrive/EdgeLLM-Systems')
MODEL_DIR = PROJECT_DIR / 'models'
RESULT_DIR = PROJECT_DIR / 'results' / 'exp001B' / 'temp'
CSV_DIR = RESULT_DIR / 'csv'
FIGURE_DIR = RESULT_DIR / 'figures'
EXP_INFO_DIR = RESULT_DIR / 'exp_info'
RUN_CONFIG_DIR = EXP_INFO_DIR / 'run_config'
ENV_INFO_DIR = EXP_INFO_DIR / 'environment'
MODEL_INFO_DIR = EXP_INFO_DIR / 'model'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
ENV_INFO_DIR.mkdir(parents=True, exist_ok=True)
MODEL_INFO_DIR.mkdir(parents=True, exist_ok=True)

if not (PROJECT_DIR / 'edge_llm_systems').exists():
    raise FileNotFoundError(
        f'Cannot find project package at {PROJECT_DIR / "edge_llm_systems"}. '
        'Please sync the repository to MyDrive/EdgeLLM-Systems first.'
    )

# 中文注释：把当前工作目录固定到项目根目录，后续相对路径、导入和调试输出都更稳定。
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('PROJECT_DIR:    ', PROJECT_DIR)
print('CWD:            ', Path.cwd())
print('MODEL_DIR:      ', MODEL_DIR)
print('CSV_DIR:        ', CSV_DIR)
print('FIGURE_DIR:     ', FIGURE_DIR)
print('EXP_INFO_DIR:   ', EXP_INFO_DIR)
print('RUN_CONFIG_DIR: ', RUN_CONFIG_DIR)
print('ENV_INFO_DIR:   ', ENV_INFO_DIR)
print('MODEL_INFO_DIR: ', MODEL_INFO_DIR)

## 3. Dependencies

安装运行 notebook 所需依赖。v2.2 追加 `ipywidgets`、`psutil`、`bitsandbytes` 和 `accelerate`，分别用于交互控件、CPU 内存统计、INT8/INT4 量化加载和自动 device map。

In [ ]:
#@title 3.1 Install / Check Dependencies
# 中文注释：追加 ipywidgets / psutil / bitsandbytes / accelerate，用于交互配置、CPU RSS 内存统计和 GPU 量化加载。
!pip install -q transformers accelerate sentencepiece huggingface_hub pyyaml pandas matplotlib ipywidgets psutil bitsandbytes

## 4. Hugging Face Access

Gemma weights may require Hugging Face access approval. This cell reads the token from Colab Secrets instead of prompting for manual input. Recommended secret name: `HF_TOKEN`.

In [ ]:
#@title 4.1 Hugging Face Access from Colab Secrets
use_hf_token_from_secrets = True #@param {type:"boolean"}
hf_secret_name = "HF_TOKEN" #@param {type:"string"}

if use_hf_token_from_secrets:
    try:
        from google.colab import userdata
        from huggingface_hub import login

        hf_token = userdata.get(hf_secret_name)
        if hf_token:
            login(token=hf_token)
            print(f'Hugging Face login completed from Colab Secret: {hf_secret_name}')
        else:
            print(f'No token found in Colab Secret: {hf_secret_name}. Cached models may still load.')
    except Exception as exc:
        print('Hugging Face secret login skipped or failed:', exc)
else:
    print('HF login skipped. Cached models can still be loaded from MODEL_DIR.')

## 5. Imports And Environment

导入依赖并记录运行环境。v2.2 在这里新增 `psutil`、`AutoModelForCausalLM` 和 `AutoTokenizer`，为 CPU 内存指标和自定义加载分支做准备。

In [ ]:
#@title 5.1 Import Modules, Runtime Adapter, and Record Environment
# 中文注释：本 cell 导入模块、记录环境，并定义设备自适应的 measure_single_runtime。
# 设计意图：不修改 benchmark 脚本本体，只在 notebook 层增加 GPU/CPU 指标适配。
import json
import os
import platform
import time
from pathlib import Path

import pandas as pd
import psutil
import torch
import transformers
import yaml
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

import benchmarks.profiling.run_exp001_v2_1 as exp001_v2_1
from benchmarks.profiling.run_exp001_v2_1 import (
    build_test_configs,
    choose_warmup_gen_len,
    make_load_status_row,
    run_matrix,
    write_csv,
)
from edge_llm_systems.cuda_utils import cleanup_cuda, require_cuda
from edge_llm_systems.kv_cache import estimate_kv_cache_mb, kv_cache_size_from_past_key_values_mb
from edge_llm_systems.metrics import tokens_per_second
from edge_llm_systems.model_registry import get_model_spec
from edge_llm_systems.model_utils import inspect_causal_lm
from edge_llm_systems.prompts import build_prompt_inputs
from scripts.plot_exp001B_v2_1 import plot_exp001B_v2_1

# 中文注释：如果用户调整控件后忘了再次点击按钮，这里同步一次，避免后续读取旧变量。
if 'sync_config_from_widgets' in globals():
    sync_config_from_widgets(show=False)

# 中文注释：兜底初始化模型加载状态。若 7.1 未运行或中途失败，8.1/9.1 会安全跳过而不是 NameError。
if 'MODEL_LOADED' not in globals():
    MODEL_LOADED = False
if 'model' not in globals():
    model = None
if 'tokenizer' not in globals():
    tokenizer = None
if 'device' not in globals():
    device = torch.device('cuda') if DEVICE_MODE == 'gpu' else torch.device('cpu')


def measure_single_runtime(prompt_len, gen_len, model, tokenizer, device, base_prompt):
    # 中文注释：该函数保留原有手动 decode 流程，只切换 GPU/CPU 的 peak_mem_mb 统计方式。
    inputs = build_prompt_inputs(tokenizer, prompt_len, device, base_prompt)
    actual_prompt_len = inputs.input_ids.shape[1]

    if DEVICE_MODE == 'gpu':
        # 中文注释：GPU 模式保持原有 CUDA 同步和 peak memory 统计逻辑。
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    else:
        # 中文注释：CPU 没有 CUDA peak counter，用当前 Python 进程 RSS 近似表示内存压力。
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss / 1024**2

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model(input_ids=inputs.input_ids, use_cache=True)
    if DEVICE_MODE == 'gpu':
        torch.cuda.synchronize()
    ttft_ms = (time.perf_counter() - t0) * 1000

    past_kv = outputs.past_key_values
    kv_pkv_prefill_mb = kv_cache_size_from_past_key_values_mb(past_kv)
    next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    decode_times = []
    out = None
    actual_gen_len = 0
    for _ in range(gen_len):
        t1 = time.perf_counter()
        with torch.no_grad():
            out = model(
                input_ids=next_token,
                past_key_values=past_kv,
                use_cache=True,
            )
        if DEVICE_MODE == 'gpu':
            torch.cuda.synchronize()
        decode_times.append((time.perf_counter() - t1) * 1000)
        actual_gen_len += 1
        past_kv = out.past_key_values
        next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)

    kv_pkv_final_mb = kv_cache_size_from_past_key_values_mb(past_kv)
    tpot_ms = sum(decode_times) / len(decode_times)
    if DEVICE_MODE == 'gpu':
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2
    else:
        mem_after = process.memory_info().rss / 1024**2
        peak_mem_mb = mem_after
    kv_est_mb = estimate_kv_cache_mb(model, actual_prompt_len + actual_gen_len)
    kv_peak_pct = kv_pkv_final_mb / peak_mem_mb * 100 if peak_mem_mb else 0.0

    result = {
        'prompt_len': actual_prompt_len,
        'requested_gen_len': gen_len,
        'actual_gen_len': actual_gen_len,
        'ttft_ms': round(ttft_ms, 2),
        'tpot_ms': round(tpot_ms, 2),
        'tokens_s': round(tokens_per_second(tpot_ms), 1),
        'peak_mem_mb': round(peak_mem_mb, 1),
        'kv_est_mb': round(kv_est_mb, 2),
        'kv_pkv_prefill_mb': round(kv_pkv_prefill_mb, 2),
        'kv_pkv_final_mb': round(kv_pkv_final_mb, 2),
        'kv_peak_pct': round(kv_peak_pct, 2),
    }

    del inputs, outputs, out, past_kv, next_token
    cleanup_cuda()
    return result


cuda_available = torch.cuda.is_available()
environment_info = {
    'project_dir': str(PROJECT_DIR),
    'model_dir': str(MODEL_DIR),
    'csv_dir': str(CSV_DIR),
    'figure_dir': str(FIGURE_DIR),
    'exp_info_dir': str(EXP_INFO_DIR),
    'run_config_dir': str(RUN_CONFIG_DIR),
    'environment_dir': str(ENV_INFO_DIR),
    'model_info_dir': str(MODEL_INFO_DIR),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'pytorch': torch.__version__,
    'transformers': transformers.__version__,
    'pandas': pd.__version__,
    'psutil': psutil.__version__,
    'cuda_available': cuda_available,
    'cuda_runtime': torch.version.cuda,
    'device_mode': DEVICE_MODE,
    'requested_precision': PRECISION,
}

if cuda_available:
    props = torch.cuda.get_device_properties(0)
    environment_info.update({
        'gpu': torch.cuda.get_device_name(0),
        'cuda_capability': torch.cuda.get_device_capability(0),
        'gpu_memory_gb': round(props.total_memory / 1024**3, 2),
    })

for key, value in environment_info.items():
    print(f'{key}: {value}')

# 中文注释：GPU 模式必须检查 CUDA；CPU 模式允许无 CUDA，但会非常慢。
if DEVICE_MODE == 'gpu':
    require_cuda()
else:
    print('CPU Only 模式：跳过 CUDA 强制检查。⚠️ CPU模式通常会比GPU慢很多。')

## 6. Runtime Config

把交互式配置转换为可保存、可复现的运行配置，并生成不会覆盖旧文件的 CSV/PNG 路径。

In [ ]:
#@title 6.1 Build v2.2 Runtime Config
# 中文注释：本 cell 把交互式控件值固化为运行配置，并生成不覆盖旧文件的输出路径。
from datetime import datetime

if 'sync_config_from_widgets' in globals():
    sync_config_from_widgets(show=False)

if not CONFIG_CONFIRMED:
    raise RuntimeError('请先在 Cell 1.1 点击“✅ 确认配置”。')
if not prompt_lengths:
    raise ValueError('请至少选择一个 prompt_len。')
if not gen_lengths:
    raise ValueError('请至少选择一个 gen_len。')

selected_model = get_model_spec(model_choice)
model_slug = selected_model.slug
run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')

# 中文注释：CPU 模式下 FP16/INT8/INT4 都会以 FP32 实际加载，因此文件名使用 cpu_fp32，避免误解实验精度。
EFFECTIVE_PRECISION = 'fp32' if DEVICE_MODE == 'cpu' else PRECISION
base_output_stem = f'exp001_results_{DEVICE_MODE}_{EFFECTIVE_PRECISION}'


def make_non_overwrite_path(directory, stem, suffix):
    # 中文注释：如果标准文件名已存在，则自动追加时间戳，保证不覆盖任何已有实验结果。
    path = directory / f'{stem}{suffix}'
    if not path.exists():
        return path
    return directory / f'{stem}_{run_tag}{suffix}'


output_csv = make_non_overwrite_path(CSV_DIR, base_output_stem, '.csv')
output_figure = make_non_overwrite_path(FIGURE_DIR, base_output_stem, '.png')
run_config_path = RUN_CONFIG_DIR / f'{base_output_stem}_{run_tag}_run_config.json'
environment_info_path = ENV_INFO_DIR / f'{base_output_stem}_{run_tag}_environment.json'
model_info_path = MODEL_INFO_DIR / f'{model_slug}_model_info.json'

config = {
    'experiment_name': 'exp001B_v2.2_multi_device_precision_probe',
    'model_choice': model_choice,
    'model_id': selected_model.model_id,
    'model_slug': model_slug,
    'model_dir': str(MODEL_DIR),
    'device_mode': DEVICE_MODE,
    'requested_precision': PRECISION,
    'effective_precision': EFFECTIVE_PRECISION,
    'prompt_lengths': list(prompt_lengths),
    'gen_lengths': list(gen_lengths),
    'repeat': int(REPEAT),
    'base_prompt': 'The quick brown fox jumps over the lazy dog. ',
    'output_csv': str(output_csv),
    'output_figure': str(output_figure),
    'run_config_path': str(run_config_path),
    'environment_info_path': str(environment_info_path),
    'model_info_path': str(model_info_path),
}
run_config_record = {
    **config,
    'matrix_configs': build_test_configs(config['prompt_lengths'], config['gen_lengths']),
}
environment_record = {
    **environment_info,
    'experiment_name': config['experiment_name'],
    'model_choice': config['model_choice'],
    'model_id': config['model_id'],
    'model_slug': config['model_slug'],
    'run_tag': run_tag,
    'output_csv': str(output_csv),
    'output_figure': str(output_figure),
}

with run_config_path.open('w', encoding='utf-8') as f:
    json.dump(run_config_record, f, ensure_ascii=False, indent=2)
with environment_info_path.open('w', encoding='utf-8') as f:
    json.dump(environment_record, f, ensure_ascii=False, indent=2)

print('Selected model:', selected_model.choice)
print('Hugging Face model_id:', selected_model.model_id)
print('Family:', selected_model.family)
print('Size label:', selected_model.size_label)
print('Notes:', selected_model.notes)
print('Device mode:', config['device_mode'])
print('Requested precision:', config['requested_precision'])
print('Effective precision:', config['effective_precision'])
print('Prompt lengths:', config['prompt_lengths'])
print('Generation lengths:', config['gen_lengths'])
print('Matrix configs:', build_test_configs(config['prompt_lengths'], config['gen_lengths']))
print('Repeat:', config['repeat'])
print('Output CSV:', output_csv)
print('Output figure:', output_figure)
print('Run config JSON:', run_config_path)
print('Environment JSON:', environment_info_path)
print('Model info JSON:', model_info_path)
if DEVICE_MODE == 'cpu':
    print('⚠️ CPU模式下建议先用小矩阵（prompt=[64], gen=[32]）验证跑通。')

## 7. Model Loading

根据 `DEVICE_MODE` 和 `PRECISION` 自动选择模型加载方式。GPU 模式支持 FP16 / INT8 / INT4；CPU 模式统一以 FP32 加载。

In [ ]:
#@title 7.1 Load Model, Inspect Metadata, and Save Model Info
# 中文注释：本 cell 是加载层重构的核心，根据 DEVICE_MODE + PRECISION 自动选择加载策略。
# 设计意图：把不同设备和精度的差异限制在加载层，避免污染后续 profiling 流程。
MODEL_LOADED = False
model_info = {}
model = None
tokenizer = None
device = torch.device('cuda') if DEVICE_MODE == 'gpu' else torch.device('cpu')
model_path = None
LOAD_OOM_MESSAGE_ZH = '模型加载阶段发生 CUDA 显存不足；本次不自动降级模型或缩小 prompt_len，直接记录为 OOM。'
LOAD_ERROR_PREFIX_ZH = '模型加载失败：'


def resolve_model_path(model_id, model_dir):
    # 中文注释：优先复用 Google Drive 中已下载的模型，避免每次 Colab runtime 重复下载权重。
    model_name = model_id.split('/')[-1]
    local_path = Path(model_dir) / model_name
    if local_path.is_dir() and any(local_path.iterdir()):
        return local_path
    local_path.parent.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=model_id, local_dir=str(local_path))
    return local_path


def build_model_load_kwargs():
    # 中文注释：集中管理加载参数，使每个 DEVICE_MODE/PRECISION 组合的行为明确可查。
    if DEVICE_MODE == 'gpu' and PRECISION == 'fp16':
        return {
            'torch_dtype': torch.float16,
            'device_map': {'': 'cuda:0'},
        }
    if DEVICE_MODE == 'gpu' and PRECISION == 'int8':
        return {
            'load_in_8bit': True,
            'device_map': 'auto',
        }
    if DEVICE_MODE == 'gpu' and PRECISION == 'int4':
        return {
            'load_in_4bit': True,
            'device_map': 'auto',
        }
    if DEVICE_MODE == 'cpu' and PRECISION == 'fp16':
        print('⚠️ CPU不支持FP16推理，已自动降级为FP32')
        return {
            'torch_dtype': torch.float32,
            'device_map': {'': 'cpu'},
        }
    if DEVICE_MODE == 'cpu' and PRECISION in ('int8', 'int4'):
        print('⚠️ CPU量化支持有限，以FP32加载')
        return {
            'torch_dtype': torch.float32,
            'device_map': {'': 'cpu'},
        }
    raise ValueError(f'Unsupported DEVICE_MODE/PRECISION: {DEVICE_MODE}/{PRECISION}')

try:
    if DEVICE_MODE == 'gpu':
        require_cuda()
    else:
        print('CPU Only 模式：模型将加载到 CPU，运行会明显慢于 GPU。')
        if PRECISION in ('int8', 'int4'):
            print('⚠️ CPU量化支持有限，本次不会启用 bitsandbytes 量化，将以 FP32 加载。')

    model_path = resolve_model_path(config['model_id'], config['model_dir'])
    load_kwargs = build_model_load_kwargs()
    print('Model load kwargs:', load_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(model_path, **load_kwargs)
    model.eval()

    # 中文注释：量化模型可能由 Accelerate 自动放置；输入张量仍按用户选择放到主设备。
    device = torch.device('cuda') if DEVICE_MODE == 'gpu' else torch.device('cpu')
    model_info = inspect_causal_lm(model)
    model_info_record = {
        'experiment_name': config['experiment_name'],
        'model_choice': config['model_choice'],
        'model_id': config['model_id'],
        'model_slug': config['model_slug'],
        'model_path': str(model_path),
        'device_mode': config['device_mode'],
        'requested_precision': config['requested_precision'],
        'effective_precision': config['effective_precision'],
        **model_info,
    }
    MODEL_LOADED = True

    print('Model loaded:', config['model_id'])
    print('Model path:', model_path)
    print('Input device:', device)
    print('Model info:')
    for key, value in model_info_record.items():
        print(f'  {key}: {value}')

    model_info_path = Path(config['model_info_path'])
    if model_info_path.exists():
        print('Model info JSON already exists; skip saving:', model_info_path)
    else:
        model_info_path.parent.mkdir(parents=True, exist_ok=True)
        with model_info_path.open('w', encoding='utf-8') as f:
            json.dump(model_info_record, f, ensure_ascii=False, indent=2)
        print('Model info JSON saved:', model_info_path)

except torch.cuda.OutOfMemoryError:
    cleanup_cuda()
    message = LOAD_OOM_MESSAGE_ZH
    row = make_load_status_row(status='oom', message_zh=message)
    if save_results:
        write_csv(output_csv, [row])
    print(message)
    print('CSV saved:', output_csv)

except RuntimeError as exc:
    cleanup_cuda()
    if 'out of memory' in str(exc).lower():
        message = LOAD_OOM_MESSAGE_ZH
        status = 'oom'
    else:
        message = f'{LOAD_ERROR_PREFIX_ZH}{exc}'
        status = 'error'
    row = make_load_status_row(status=status, message_zh=message)
    if save_results:
        write_csv(output_csv, [row])
    print(message)
    print('CSV saved:', output_csv)

## 8. Warm-Up

对当前设备和精度进行预热。GPU 模式用于稳定 CUDA 首次调用开销；CPU 模式用于提前暴露配置错误，但会明显更慢。

In [ ]:
#@title 8.1 Warm Up Runtime
# 中文注释：预热逻辑沿用 v1/v2.1 的思路，只把 GPU 专属测量函数换成设备自适应版本。
if globals().get('MODEL_LOADED', False):
    warmup_gen_len = choose_warmup_gen_len(config['gen_lengths'])
    print(f'Warm-up gen_len: {warmup_gen_len}')
    for warmup_prompt_len in config['prompt_lengths']:
        try:
            _ = measure_single_runtime(
                warmup_prompt_len,
                warmup_gen_len,
                model,
                tokenizer,
                device,
                config['base_prompt'],
            )
            print(f'Warm-up completed: prompt={warmup_prompt_len}, gen={warmup_gen_len}')
        except torch.cuda.OutOfMemoryError:
            cleanup_cuda()
            print(f'Warm-up OOM: prompt={warmup_prompt_len}, gen={warmup_gen_len}; formal test will continue.')
        except RuntimeError as exc:
            cleanup_cuda()
            print(f'Warm-up failed: prompt={warmup_prompt_len}, gen={warmup_gen_len}; error={exc}')
else:
    print('Warm-up skipped because the model was not loaded.')

## 9. Benchmark Matrix

运行测试矩阵。v2.2 不改 `run_matrix` 的遍历、重复取均值和 OOM 记录逻辑，只在运行前把测量函数替换为 notebook 中的设备自适应版本。

In [ ]:
#@title 9.1 Run v2.2 Benchmark Matrix
# 中文注释：本 cell 不修改 run_matrix 核心逻辑，只注入设备自适应的 measure_single_runtime。
def format_row_summary(row):
    return (
        f"prompt={row['prompt_len']}, gen={row['requested_gen_len']} | "
        f"TTFT={row['ttft_ms']} ms | TPOT={row['tpot_ms']} ms | "
        f"tok/s={row['tokens_s']} | peak={row['peak_mem_mb']} MB | "
        f"KV={row['kv_pkv_final_mb']} MB | KV/peak={row['kv_peak_pct']}% | "
        f"repeat={row['repeat_completed']} | status={row['status']}"
    )


def print_progress(event):
    event_name = event.get('event')
    if event_name == 'matrix_start':
        total = event['config_total']
        repeat_total = event['repeat']
        print(f'Start matrix: {total} configs, repeat={repeat_total}, total measured runs={total * repeat_total}')
    elif event_name == 'config_start':
        print(
            f"[{event['config_index']}/{event['config_total']}] "
            f"Start config: prompt_len={event['prompt_len']}, gen_len={event['requested_gen_len']}"
        )
    elif event_name == 'repeat_start':
        print(f"  - repeat {event['repeat_index']}/{event['repeat_total']} ...")
    elif event_name == 'repeat_done':
        row = event['row']
        print(
            f"  - repeat {event['repeat_index']}/{event['repeat_total']} done: "
            f"TTFT={row['ttft_ms']} ms, TPOT={row['tpot_ms']} ms, "
            f"peak={row['peak_mem_mb']} MB, KV%={row['kv_peak_pct']}"
        )
    elif event_name == 'config_done':
        row = event['row']
        print(f"[{event['config_index']}/{event['config_total']}] Averaged result:")
        print('  ' + format_row_summary(row))
    elif event_name == 'config_oom':
        row = event['row']
        print(f"[{event['config_index']}/{event['config_total']}] OOM recorded:")
        print('  ' + format_row_summary(row))
        print('  message:', row['message_zh'])
    elif event_name == 'config_error':
        row = event['row']
        print(f"[{event['config_index']}/{event['config_total']}] ERROR recorded:")
        print('  ' + format_row_summary(row))
        print('  message:', row['message_zh'])
    elif event_name == 'matrix_done':
        print(f'Matrix done: {event["config_total"]} configs completed or recorded.')

# 中文注释：run_matrix 内部会调用模块全局 measure_single_v2_1；这里仅在 notebook 层替换为适配 CPU/GPU 的版本。
exp001_v2_1.measure_single_v2_1 = measure_single_runtime

if globals().get('MODEL_LOADED', False):
    rows = run_matrix(
        prompt_lengths=config['prompt_lengths'],
        gen_lengths=config['gen_lengths'],
        model=model,
        tokenizer=tokenizer,
        device=device,
        repeat=config['repeat'],
        base_prompt=config['base_prompt'],
        progress_callback=print_progress,
    )

    if save_results:
        write_csv(output_csv, rows)
        print('CSV saved:', output_csv)

    df = pd.DataFrame(rows)
    display(df)
else:
    print('Model was not loaded. Check Cell 7.1 output and the saved load-status CSV if available.')

## 10. Plot Results

Generate a v1-style overview figure from the saved CSV. Only `status=ok` rows are plotted; OOM rows remain in the CSV.

In [ ]:
#@title 10.1 Plot v2.1 Results
if save_results and 'output_csv' in globals() and output_csv.exists():
    plot_exp001B_v2_1(
        csv_path=output_csv,
        output_path=output_figure,
        model_label=config['model_choice'],
    )
    print('Figure saved:', output_figure)
else:
    print('No saved CSV found. Run Cell 9.0 with save_results=True before plotting.')

## 11. Cleanup

释放运行时缓存。GPU 模式会释放 CUDA cache；CPU 模式主要依赖 Python GC。

In [ ]:
#@title 11.1 Release Runtime Cache
# 中文注释：清理运行时缓存，避免后续实验继承当前模型对象和 CUDA cache。
if 'model' in globals():
    try:
        del model
    except Exception:
        pass
if 'cleanup_cuda' in globals():
    cleanup_cuda()
else:
    print('cleanup_cuda is not available; skip runtime cleanup.')
print('Runtime cleanup completed.')